## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings('ignore')

## Load Cleaned Dataset

In [2]:
# Load cleaned dataset
df = pd.read_csv('../data/data_cleaned.csv')
df_original = df.copy()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset shape: (45194, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


## Create Derived Features

In [3]:
df['age_group'] = pd.cut(df['age'], 
                        bins=[0, 25, 35, 45, 55, 100], 
                        labels=['Young', 'Adult', 'Middle', 'Senior', 'Elder'])

In [4]:
df['overtime'] = (df['hours-per-week'] > 40).astype(int)
df['work_intensity'] = pd.cut(df['hours-per-week'], 
                             bins=[0, 20, 40, 50, 100], 
                             labels=['Part-time', 'Standard', 'Over-time', 'Extreme'])

In [5]:
df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)
df['has_capital_loss'] = (df['capital-loss'] > 0).astype(int)
df['capital_net'] = df['capital-gain'] - df['capital-loss']

In [6]:
education_groups = {
    'Low': ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th'],
    'Medium': ['HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm'], 
    'High': ['Bachelors', 'Masters', 'Prof-school', 'Doctorate']
}

def map_education_group(education):
    for group, educations in education_groups.items():
        if education in educations:
            return group
    return 'Other'

df['education_group'] = df['education'].apply(map_education_group)


In [7]:
married_status = ['Married-civ-spouse', 'Married-spouse-absent', 'Married-AF-spouse']
df['is_married'] = df['marital-status'].isin(married_status).astype(int)

In [8]:
df['is_us_native'] = (df['native-country'] == 'United-States').astype(int)

In [9]:
df.shape

(45194, 24)

## Create Interaction Features

In [10]:
# 1. Age × Education interaction (experience with education level)
df['age_education_interaction'] = df['age'] * df['education-num']

# 2. Hours × Education interaction (work commitment with education)
df['hours_education_interaction'] = df['hours-per-week'] * df['education-num']

# 3. Age × Hours interaction (experience with work commitment)
df['age_hours_interaction'] = df['age'] * df['hours-per-week']

# 4. Marriage × Education interaction
df['marriage_education_interaction'] = df['is_married'] * df['education-num']

In [11]:
interaction_features = ['age_education_interaction', 'hours_education_interaction', 
                       'age_hours_interaction', 'marriage_education_interaction']
print(f"Interaction features: {interaction_features}")

Interaction features: ['age_education_interaction', 'hours_education_interaction', 'age_hours_interaction', 'marriage_education_interaction']


In [12]:
df.shape

(45194, 28)

## Encode Categorical Variables

In [13]:
# Identify categorical columns (exclude target)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_to_encode = [col for col in categorical_cols if col != 'income']

print(f"Categorical columns to encode: {categorical_to_encode}")

Categorical columns to encode: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'education_group']


In [14]:
# Label Encoding for binary/ordinal columns
binary_cols = ['sex']  # Binary categories
label_encoders = {}

print("\nLabel encoding binary columns:")
for col in binary_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col])
        label_encoders[col] = le
        print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Label encoding binary columns:
  sex: {'Female': np.int64(0), 'Male': np.int64(1)}


In [15]:
# One-Hot Encoding for nominal categorical columns
nominal_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 
                'race', 'native-country', 'education', 'age_group', 'work_intensity', 
                'education_group']
nominal_cols = [col for col in nominal_cols if col in df.columns]

print(f"\nOne-hot encoding nominal columns: {nominal_cols}")


One-hot encoding nominal columns: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'native-country', 'education', 'age_group', 'work_intensity', 'education_group']


In [16]:
# Create one-hot encoded features
df_encoded = pd.get_dummies(df, columns=nominal_cols, prefix_sep='_', drop_first=True, dtype=int)

In [17]:
print(f"Dataset shape after encoding: {df_encoded.shape}")
print(f"Columns expanded from {df.shape[1]} to {df_encoded.shape[1]}")

Dataset shape after encoding: (45194, 117)
Columns expanded from 29 to 117


In [18]:
# Show sample of new columns
new_columns = [col for col in df_encoded.columns if col not in df.columns]
new_columns[:10]

['workclass_Local-gov',
 'workclass_Private',
 'workclass_Self-emp-inc',
 'workclass_Self-emp-not-inc',
 'workclass_State-gov',
 'workclass_Without-pay',
 'marital-status_Married-AF-spouse',
 'marital-status_Married-civ-spouse',
 'marital-status_Married-spouse-absent',
 'marital-status_Never-married']

## Prepare Target Variable

In [19]:
# Encode target variable
target_encoder = LabelEncoder()
df_encoded['income_target'] = target_encoder.fit_transform(df_encoded['income'])

In [20]:
# Show encoding mapping
target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print(f"Target encoding: {target_mapping}")

Target encoding: {'<=50K': np.int64(0), '>50K': np.int64(1)}


In [21]:
# Verify target distribution
target_dist = df_encoded['income_target'].value_counts()
print(f"\nTarget distribution:")
for value, count in target_dist.items():
    label = target_encoder.inverse_transform([value])[0]
    percentage = (count / len(df_encoded)) * 100
    print(f"  {label} ({value}): {count:,} ({percentage:.1f}%)")


Target distribution:
  <=50K (0): 33,988 (75.2%)
  >50K (1): 11,206 (24.8%)


## Scale Numerical Features

In [22]:
# Identify numerical columns to scale (exclude target and binary encoded)
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 
                     'hours-per-week', 'education_rank', 'capital_net'] + interaction_features

# Filter existing columns
numerical_features = [col for col in numerical_features if col in df_encoded.columns]
print(f"Numerical features to scale: {numerical_features}")

Numerical features to scale: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'capital_net', 'age_education_interaction', 'hours_education_interaction', 'age_hours_interaction', 'marriage_education_interaction']


In [23]:
# Initialize scaler
scaler = StandardScaler()

# Create scaled dataset
df_scaled = df_encoded.copy()
df_scaled[numerical_features] = scaler.fit_transform(df_scaled[numerical_features])

print(f"\nNumerical features scaled using StandardScaler")
print(f"Features scaled: {len(numerical_features)}")


Numerical features scaled using StandardScaler
Features scaled: 11


In [24]:
# Show scaling statistics
print(f"\nScaling verification (mean ≈ 0, std ≈ 1):")
scaling_stats = df_scaled[numerical_features].agg(['mean', 'std']).round(3)
scaling_stats.head()


Scaling verification (mean ≈ 0, std ≈ 1):


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,capital_net,age_education_interaction,hours_education_interaction,age_hours_interaction,marriage_education_interaction
mean,-0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Feature Selection and Final Dataset

In [25]:
# Remove original categorical columns and redundant features
columns_to_drop = [
    'income',  # Original target (use income_target)
    'fnlwgt',  # Not relevant for prediction
    'education-num',  # Redundant with education_rank
]

In [26]:
# Add original categorical columns that were encoded
original_categorical = ['workclass', 'education', 'marital-status', 'occupation', 
                       'relationship', 'race', 'sex', 'native-country']
columns_to_drop.extend([col for col in original_categorical if col in df_scaled.columns])

# Remove columns
df_final = df_scaled.drop(columns=[col for col in columns_to_drop if col in df_scaled.columns])

print(f"Columns removed: {[col for col in columns_to_drop if col in df_scaled.columns]}")

Columns removed: ['income', 'fnlwgt', 'education-num', 'sex']


In [27]:
print(f"Final dataset shape: {df_final.shape}")

Final dataset shape: (45194, 114)


In [28]:
df_final.head()

,age,capital-gain,capital-loss,hours-per-week,overtime,has_capital_gain,has_capital_loss,capital_net,is_married,is_us_native,...,age_group_Adult,age_group_Middle,age_group_Senior,age_group_Elder,work_intensity_Standard,work_intensity_Over-time,work_intensity_Extreme,education_group_Low,education_group_Medium,income_target
0,0.033918,0.142754,-0.218851,-0.078393,0,1,0,0.154070,0,1,...,0,1,0,0,1,0,0,0,0,0
1,0.866264,-0.146780,-0.218851,-2.327209,0,0,0,-0.134545,1,1,...,0,0,1,0,0,0,0,0,0,0
2,-0.041750,-0.146780,-0.218851,-0.078393,0,0,0,-0.134545,0,1,...,0,1,0,0,1,0,0,0,1,0
3,1.093267,-0.146780,-0.218851,-0.078393,0,0,0,-0.134545,1,1,...,0,0,1,0,1,0,0,1,0,0
4,-0.798428,-0.146780,-0.218851,-0.078393,0,0,0,-0.134545,1,0,...,1,0,0,0,1,0,0,0,0,0


## Save Processed Data

In [29]:
# Save complete processed dataset
df_final.to_csv('../data/data_features_engineering.csv', index=False)

In [30]:
# Save encoders for future use
import joblib

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(target_encoder, '../models/target_encoder.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl')

print("\nEncoders saved:")
print("  - scaler.pkl")
print("  - target_encoder.pkl")
print("  - label_encoders.pkl")


Encoders saved:
  - scaler.pkl
  - target_encoder.pkl
  - label_encoders.pkl
